# Comparative Tree Analysis: Structure vs Sequence-Based 3Di

This notebook compares phylogenetic trees built using:
1. **Ground truth 3Di** - Extracted from protein structures using FoldSeek
2. **Predicted 3Di** - Inferred from amino acid sequences using ESM3Di model
3. **Traditional amino acid sequences**

## Tree Building Methods
- **Maximum Likelihood (RAxML)** - More accurate, slower
- **Distance-based (QuickTree)** - Faster, useful for large datasets

## Requirements
- Input: Directory of PDB/mmCIF structure files
- Tools: FoldSeek, MAFFT, RAxML-ng, QuickTree
- ESM3Di checkpoint for 3Di prediction

In [1]:
# Core imports
import os
import subprocess
import tempfile
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from tqdm import tqdm
import torch
import numpy as np

# Import tree building and foldseek utilities from esm3di
from esm3di import (
    ESM3DiModel,
    predict_3di_from_sequences,
    run_mafft,
    run_raxml,
    run_quicktree,
    run_foldseek_createdb,
    run_foldseek_allvall,
    read_foldseek_db,
    tajima_distance,
    write_distance_matrix,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/home/dmoi/miniforge3/envs/fastas2foldseekdb/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 4000 Blackwell


## 2. Configuration

In [12]:
@dataclass
class Config:
    """Configuration for the analysis pipeline."""
    # Input
    structure_dir: str = "structures"  # Directory containing PDB/mmCIF files
    
    # Output
    output_dir: str = "tree_analysis_output"
    hf_model_name: str = "Synthyra/ESMplusplus_small"  # Hugging Face model name for ESM3Di
    # ESM3Di model settings
    esm3di_checkpoint: str =  "./checkpoints_esmpp_bfvd/epoch_3.pt"
    batch_size: int = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Tool paths (modify if not in PATH)
    foldseek_bin: str = "foldseek"
    mafft_bin: str = "mafft"
    raxml_bin: str = "raxml-ng"
    quicktree_bin: str = "quicktree"

# Create config instance
config = Config()

# Create output directory
os.makedirs(config.output_dir, exist_ok=True)
print(f"Output directory: {config.output_dir}")
print(f"Device: {config.device}")

Output directory: tree_analysis_output
Device: cuda


In [13]:
# Configure input/output
STRUCTURE_DIR = "/mnt/data2/Mifsud_et_al_Repository/glycoprotein_structural_alignments_and_trees/refolded_fullglyco"  # <-- Change this to your structure directory
OUTPUT_DIR = "tree_analysis_output"
CHECKPOINT = "./checkpoints_esmpp_bfvd/epoch_3.pt"  # <-- ESM3Di checkpoint (optional)

# Verify input directory exists
if not os.path.exists(STRUCTURE_DIR):
    print(f"⚠️ Structure directory not found: {STRUCTURE_DIR}")
    print("Please set STRUCTURE_DIR to a directory containing PDB files.")
else:
    pdb_files = list(Path(STRUCTURE_DIR).glob("*.pdb")) + list(Path(STRUCTURE_DIR).glob("*.cif"))
    print(f"Found {len(pdb_files)} structure files in {STRUCTURE_DIR}")

Found 626 structure files in /mnt/data2/Mifsud_et_al_Repository/glycoprotein_structural_alignments_and_trees/refolded_fullglyco


In [14]:
#create the amino acid and ground truth 3Di state mapping
outdb = run_foldseek_createdb(STRUCTURE_DIR, os.path.join(OUTPUT_DIR, "foldseek_db"), config.foldseek_bin)
mapAA , map3Di = read_foldseek_db(db=outdb)
print(f"Mapped {len(mapAA)} amino acids and {len(map3Di)} 3Di states from Foldseek database.")

  Command: foldseek createdb /mnt/data2/Mifsud_et_al_Repository/glycoprotein_structural_alignments_and_trees/refolded_fullglyco tree_analysis_output/foldseek_db/foldseek
✓ FoldSeek database created at tree_analysis_output/foldseek_db/foldseek
Mapped 626 amino acids and 626 3Di states from Foldseek database.


In [15]:
#output the mapping files to fasta files for reference
with open(os.path.join(OUTPUT_DIR, "amino_acid_mapping.fasta"), "w") as aa_fasta:
	for aa, seq in mapAA.items():
		record = SeqRecord(Seq(seq), id=aa, description="")
		SeqIO.write(record, aa_fasta, "fasta")
with open(os.path.join(OUTPUT_DIR, "3di_state_mapping.fasta"), "w") as threedi_fasta:
	for state, seq in map3Di.items():
		record = SeqRecord(Seq(seq), id=state, description="")
		SeqIO.write(record, threedi_fasta, "fasta")

In [20]:
#infer the 3di states for all sequences in using the ESM3Di model
predicted_3di = predict_3di_from_sequences(os.path.join(OUTPUT_DIR, "amino_acid_mapping.fasta"), output_fasta=os.path.join(OUTPUT_DIR, "predicted_3di.fasta"), checkpoint_path=config.esm3di_checkpoint, batch_size=config.batch_size, device=config.device)
print(f"Predicted 3Di states for {len(predicted_3di)} sequences using ESM3Di model.")

Loading ESM3Di model from ./checkpoints_esmpp_bfvd/epoch_3.pt...


Some weights of EsmForTokenClassification were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Base model: facebook/esm2_t12_35M_UR50D
  Labels: 20
  Device: cuda

Loading model: facebook/esm2_t12_35M_UR50D
✓ Base model loaded (TokenClassification)

Auto-discovering LoRA target modules...
Discovered target modules: ['esm.embeddings.position_embeddings', 'esm.embeddings.word_embeddings', 'esm.encoder.layer.0.attention.output.dense', 'esm.encoder.layer.0.attention.self.key', 'esm.encoder.layer.0.attention.self.query', 'esm.encoder.layer.0.attention.self.value', 'esm.encoder.layer.0.intermediate.dense', 'esm.encoder.layer.0.output.dense', 'esm.encoder.layer.1.attention.output.dense', 'esm.encoder.layer.1.attention.self.key', 'esm.encoder.layer.1.attention.self.query', 'esm.encoder.layer.1.attention.self.value', 'esm.encoder.layer.1.intermediate.dense', 'esm.encoder.layer.1.output.dense', 'esm.encoder.layer.10.attention.output.dense', 'esm.encoder.layer.10.attention.self.key', 'esm.encoder.layer.10.attention.self.query', 'esm.encoder.layer.10.attention.self.value', 'esm.encoder.la

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Reading input FASTA: tree_analysis_output/amino_acid_mapping.fasta
Found 626 sequences
Processed 40/626 sequences
Processed 80/626 sequences
Processed 120/626 sequences
Processed 160/626 sequences
Processed 200/626 sequences
Processed 240/626 sequences
Processed 280/626 sequences
Processed 320/626 sequences
Processed 360/626 sequences
Processed 400/626 sequences
Processed 440/626 sequences
Processed 480/626 sequences
Processed 520/626 sequences
Processed 560/626 sequences
Processed 600/626 sequences
Processed 626/626 sequences
Writing predictions to: tree_analysis_output/predicted_3di.fasta
✓ Prediction complete! 626 sequences written to tree_analysis_output/predicted_3di.fasta

✓ Predicted 3Di written to tree_analysis_output/predicted_3di.fasta
Predicted 3Di states for 40 sequences using ESM3Di model.


In [22]:
#count number of sequences in the predicted 3Di fasta file
predicted_3di_count = len(list(SeqIO.parse(os.path.join(OUTPUT_DIR, "predicted_3di.fasta"), "fasta")))
print(f"Number of sequences in predicted 3Di fasta file: {predicted_3di_count}")

Number of sequences in predicted 3Di fasta file: 626


In [ ]:
#create foldseek database for the predicted 3Di sequences
from esm3di.fastas2foldseekdb import create_foldseek_db_from_fastas
import shutil
predicted_db_folder = os.path.join(OUTPUT_DIR, "predicted_foldseek_db")
os.makedirs(predicted_db_folder, exist_ok=True)
#copy the amino acid mapping fasta to the predicted foldseek db folder for reference
shutil.copy(os.path.join(OUTPUT_DIR, "amino_acid_mapping.fasta"), predicted_db_folder)
#copy the predicted 3Di fasta to the predicted foldseek db folder for reference
shutil.copy(os.path.join(OUTPUT_DIR, "predicted_3di.fasta"), predicted_db_folder)
predicted_db = create_foldseek_db_from_fastas(
    aa_fasta=os.path.join(predicted_db_folder, "amino_acid_mapping.fasta"),
    three_di_fasta=os.path.join(predicted_db_folder, "predicted_3di.fasta"),
    output_db=os.path.join(predicted_db_folder, "predicted_db"),
    foldseek_bin="foldseek"
)
print(f"Created Foldseek database for predicted 3Di sequences at: {predicted_db}")

Running: foldseek tsv2db /tmp/tmpmgetqp9m_aa.tsv tree_analysis_output/predicted_foldseek_db/predicted_db --output-dbtype 0
tsv2db /tmp/tmpmgetqp9m_aa.tsv tree_analysis_output/predicted_foldseek_db/predicted_db --output-dbtype 0 

MMseqs Version:           	10.941cd33
Output database type      	0
Compressed                	0
Verbosity                 	3

Output database type: Aminoacid
Time for merging to predicted_db: 0h 0m 0s 0ms
Time for processing: 0h 0m 0s 2ms

Running: foldseek tsv2db /tmp/tmppcqfqlqv_3di.tsv tree_analysis_output/predicted_foldseek_db/predicted_db_ss --output-dbtype 0
tsv2db /tmp/tmppcqfqlqv_3di.tsv tree_analysis_output/predicted_foldseek_db/predicted_db_ss --output-dbtype 0 

MMseqs Version:           	10.941cd33
Output database type      	0
Compressed                	0
Verbosity                 	3

Output database type: Aminoacid
Time for merging to predicted_db_ss: 0h 0m 0s 0ms
Time for processing: 0h 0m 0s 2ms

Running: foldseek tsv2db /tmp/tmpl6uf8867_header.

In [27]:
#run all-vs-all Foldseek search to get pairwise distances on predicted db and ground truth db
allvall_output = os.path.join(predicted_db_folder, "predicted_allvall.tsv")
run_foldseek_allvall(predicted_db, predicted_db, allvall_output, config.foldseek_bin)
print(f"Completed all-vs-all Foldseek search. Output saved to: {allvall_output}")

allvall_output_ground_truth = os.path.join(OUTPUT_DIR, "ground_truth_allvall.tsv")
run_foldseek_allvall(outdb, outdb, allvall_output_ground_truth, config.foldseek_bin)
print(f"Completed all-vs-all Foldseek search on ground truth db. Output saved to: {allvall_output_ground_truth}")

TypeError: sequence item 2: expected str instance, bool found